In [ ]:
from sampo.hybrid.population import PopulationScheduler
import numpy as np
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import sampo.scheduler

from sampo.userinput.parser.csv_parser import CSVParser
from sampo.hybrid.cycle import CycleHybridScheduler
from sampo.api.genetic_api import ScheduleGenerationScheme, Individual
from sampo.scheduler import HEFTScheduler, HEFTBetweenScheduler, TopologicalScheduler, GeneticScheduler
from sampo.hybrid.population import HeuristicPopulationScheduler, GeneticPopulationScheduler

from sampo.generator.environment import get_contractor_by_wg
from sampo.generator import SimpleSynthetic

from sampo.base import SAMPO
from sampo.schemas import WorkGraph
from sampo.schemas.time import Time

# Импорт ресурсной модели
from models.field_dev_resources_time_estimator import FieldDevWorkEstimator

# Фитнесс-функции
from sampo.scheduler.genetic.operators import TimeFitness, DeadlineCostFitness, TimeAndResourcesFitness, TimeAndCostFitness

# Потом можно заменить, если добавить в функциях аргументы по умолчанию
from sampo.schemas.landscape import LandscapeConfiguration
from sampo.schemas.schedule_spec import ScheduleSpec

# Визуализация
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# testing stuff
from my_modules.bee_colony import Bee, BeeColony, BeeScheduler, RandomScheduler

## Get project data

In [ ]:
DATA_PATH = Path('data/csv/gas_network_full_connections.csv')
HISTORY_PATH = Path('data/historical_projects_data.csv')

project_df = pd.read_csv(DATA_PATH, sep=',')
history_df = pd.read_csv(HISTORY_PATH, sep=';')

all_connections = True
change_connections = False

project_work_estimator = FieldDevWorkEstimator()

In [ ]:
%%capture
# not printing output

wg, contractors = CSVParser.work_graph_and_contractors(
    works_info=CSVParser.read_graph_info(
        project_info=project_df,
        history_data=history_df,
        all_connections=all_connections,
        change_connections_info=change_connections
    ),
    work_resource_estimator=project_work_estimator
)

## My scheduler

In [ ]:
fitness_object = TimeAndCostFitness()

In [ ]:
def current_single_fitness_function(bees):
    fitness_array = SAMPO.backend.compute_chromosomes(fitness_object, [
        [bee.activity_list, bee.resources_matrix, bee.ceil_list, ScheduleSpec(), np.array([])]
        for bee in bees
    ])
    return [i for i in fitness_array]

## Test

In [ ]:
# heuristics = HeuristicPopulationScheduler([
#     HEFTScheduler(work_estimator=project_work_estimator), 
#     HEFTBetweenScheduler(work_estimator=project_work_estimator), 
#     TopologicalScheduler(work_estimator=project_work_estimator)
# ])

genetic_1 = GeneticPopulationScheduler(GeneticScheduler(number_of_generation=1,
                                                        size_of_population=50,
                                                        mutate_order=0.01,
                                                        mutate_resources=0.01,
                                                        sgs_type=ScheduleGenerationScheme.Parallel,
                                                        work_estimator=project_work_estimator,
                                                        is_multiobjective=True,
                                                        fitness_weights=(-1, -1),
                                                        fitness_constructor=fitness_object))

random_scheduler = RandomScheduler(wg, contractors, current_single_fitness_function, population_size=50)
bees = BeeScheduler(wg, contractors, current_single_fitness_function)

hybrid_combine = CycleHybridScheduler(random_scheduler, [genetic_1, bees], max_plateau_size=10, max_steps=10, fitness=fitness_object)

# wg = WorkGraph.load('wgs', f'{graph_size}_{iteration}')
# contractors = [get_contractor_by_wg(wg)]

# SAMPO.backend = MultiprocessingComputationalBackend(n_cpus=10)
SAMPO.backend.cache_scheduler_info(wg, contractors, work_estimator=project_work_estimator)
SAMPO.backend.cache_genetic_info()

pop_2 = hybrid_combine.run(wg, contractors)

In [ ]:
fig = go.Figure([
    # go.Scatter(
    #     x=[ind.fitness.values[0] for ind in pop_1],
    #     y=[ind.fitness.values[1] for ind in pop_1],
    #     marker_color='green',
    #     name='Генетика',
    #     mode='markers'
    # ),
    # go.Scatter(
    #     x=[ind.fitness.values[0] for ind in pop_3],
    #     y=[ind.fitness.values[1] for ind in pop_3],
    #     marker_color='black',
    #     name='Пчелы',
    #     mode='markers'
    # ),
    go.Scatter(
        x=[ind.fitness.values[0] for ind in pop_2],
        y=[ind.fitness.values[1] for ind in pop_2],
        marker_color='navy',
        name='Генетика + Пчелы',
        mode='markers'
    ),
])
fig.update_layout(height=800, width=800, template='plotly_white')

fig.update_layout(
    xaxis_title="Время",                   
    yaxis_title="Стоимость ресурсов",
    showlegend=True,
    margin_t=0, margin_b=0, margin_l=0, margin_r=0
)

fig.update_layout(legend=dict(
    yanchor="top", y=0.95,
    xanchor="left", x=0.05
))
fig.show()

# fig.write_image('plots/multiobjective_100_iter_2.png', scale=4)